# 🚀 Thực Nghiệm Spec-fastgs Trên MipNerf360 (Tập images_8) Trên Kaggle

Notebook này tự động hóa toàn bộ các bước:
1. **Cấu hình đường dẫn** làm việc cục bộ trên Kaggle (`/kaggle/working`).
2. **Định vị & Liên kết (Symlink) dataset MipNerf360** từ Kaggle Input (`/kaggle/input/mipnerf360-dataset`) để tiết kiệm dung lượng ổ đĩa (0 bytes cho ảnh thô).
3. **Cài đặt môi trường & Biên dịch CUDA Submodules** (`diff-gaussian-rasterization_fastgs`, `simple-knn`, `fused-ssim`).
4. **Chạy batch script** huấn luyện và đánh giá trên 9 scene Mip-NeRF 360 cho tập `images_8`.
5. **Tổng hợp kết quả** từ file kết quả của từng scene thành bảng biểu dạng Excel `.xlsx` giống định dạng trong `other_job`.
6. **Đồng bộ kết quả ra thư mục output của Kaggle** để tải về trực tiếp từ giao diện UI.

## 🛠️ Bước 1: Khởi Tạo Cấu Hình & Tham Số Môi Trường

In [ ]:
# ── Setup Configs on Kaggle ───────────────────────────────────────────────────
import os
import sys

# Thư mục dự án cục bộ trên Kaggle (Không dùng Google Drive)
PROJECT_DIR = "/kaggle/working/thesis-all"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Tự động phát hiện và đọc HF_TOKEN từ Kaggle Secrets nếu có
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("🔑 Đã tự động lấy HF_TOKEN từ Kaggle User Secrets!")
except Exception:
    #@markdown **HuggingFace settings (Nếu không dùng Kaggle Secrets):**
    HF_TOKEN = "YOUR_HF_TOKEN" #@param {type:"string"}
    print("ℹ️ Sử dụng HF_TOKEN cấu hình thủ công.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print(f"✅ Cấu hình hoàn tất!")
print(f"Thư mục làm việc dự án: {PROJECT_DIR}")

## 📂 Bước 2: Git Clone Source Code về Kaggle Working Directory

In [ ]:
# ── Git Clone Repository ──────────────────────────────────────────────────────
import os
import shutil

git_dir = os.path.join(PROJECT_DIR, ".git")

if not os.path.exists(git_dir):
    if os.path.exists(PROJECT_DIR):
        # Dọn dẹp thư mục nếu nó rỗng hoặc chứa file rác để tránh xung đột
        print(f"🧹 Dọn dẹp thư mục dự án cũ...")
        shutil.rmtree(PROJECT_DIR)
    
    print(f"🔄 Đang clone repo về Kaggle Working tại {PROJECT_DIR}...")
    # Clone kèm submodules trong FastGS_backup_v2
    !git clone --recursive https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git "{PROJECT_DIR}"
    print("✅ Clone thành công!")
else:
    print(f"✅ Thư mục dự án đã tồn tại. Đang cập nhật source code mới nhất...")
    %cd "{PROJECT_DIR}"
    !git pull

## 📥 Bước 3: Tự Động Quét & Liên Kết (Symlink) Dataset MipNerf360 Từ Kaggle Input

Để tránh lỗi tràn bộ nhớ ổ đĩa `/kaggle/working` (`No space left on device`), notebook này sẽ **không** tải trực tiếp hay giải nén dataset tại `/kaggle/working`.

Thay vào đó, bạn hãy sử dụng dataset **MipNerf360-dataset** đã được tiền xử lý trước:
1. Tạo dataset **MipNerf360-dataset** bằng notebook tiền xử lý dữ liệu trước đó.
2. Add dataset **MipNerf360-dataset** vào Notebook huấn luyện này thông qua panel bên phải (`Data -> Add Input -> Select MipNerf360-dataset`).
3. Cell bên dưới sẽ tự động tìm kiếm các thư mục scene (`bicycle`, `garden`, `room`,...) trong `/kaggle/input`.
4. Tạo liên kết tượng trưng (Symlink) từ `/kaggle/input` vào `/kaggle/working/thesis-all/spec-fastgs/datasets/mipnerf360/<scene>`.
   - **Ưu điểm:** Tiết kiệm 100% dung lượng lưu trữ ảnh (tốn 0 bytes trên `/kaggle/working`), đồng thời thư mục scene trên `/kaggle/working` vẫn có tính ghi (writable) để ghi kết quả train/priors bình thường.

In [ ]:
# ── Định Vị & Liên Kết Dataset MipNerf360 Bằng Symlink ───────────────────────────
import os
import sys
import shutil
from pathlib import Path

PROJECT_DIR = "/kaggle/working/thesis-all"
dataset_target_root = os.path.join(PROJECT_DIR, "spec-fastgs", "datasets", "mipnerf360")
os.makedirs(dataset_target_root, exist_ok=True)

MIP360_SCENES = ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]

def mirror_symlink(src_abs, dst_abs):
    """Tạo cấu trúc thư mục thực sự (để có quyền ghi) và symlink các file bên trong"""
    os.makedirs(dst_abs, exist_ok=True)
    for root, dirs, files in os.walk(src_abs):
        rel_path = os.path.relpath(root, src_abs)
        current_dst = dst_abs if rel_path == "." else os.path.join(dst_abs, rel_path)
        os.makedirs(current_dst, exist_ok=True)
        for f in files:
            src_file = os.path.join(root, f)
            dst_file = os.path.join(current_dst, f)
            if os.path.exists(dst_file) or os.path.islink(dst_file):
                if os.path.isdir(dst_file) and not os.path.islink(dst_file):
                    shutil.rmtree(dst_file)
                else:
                    os.remove(dst_file)
            os.symlink(src_file, dst_file)

print("🔍 Đang quét các thư mục scene từ Kaggle Input...")
print("-" * 70)

preprocessed_found = {}
for scene in MIP360_SCENES:
    scene_src = None
    # Quét /kaggle/input để tìm thư mục scene chứa 'images_8'
    for root, dirs, files in os.walk("/kaggle/input"):
        if scene in dirs:
            test_path = os.path.join(root, scene)
            if os.path.isdir(os.path.join(test_path, "images_8")):
                scene_src = test_path
                break
    if scene_src:
        preprocessed_found[scene] = scene_src

if len(preprocessed_found) == len(MIP360_SCENES):
    print(f"✅ Đã tìm thấy đầy đủ {len(MIP360_SCENES)} scenes trong Kaggle Input!")
    print("🚀 Bắt đầu tạo liên kết (Symlink)...")
    for scene, src_path in preprocessed_found.items():
        dst_path = os.path.join(dataset_target_root, scene)
        mirror_symlink(src_path, dst_path)
        print(f"  🔗 Đã symlink scene '{scene}' từ: {src_path}")
    print("✅ Toàn bộ dataset đã được liên kết thành công và có quyền ghi tại local!")
else:
    print(f"❌ Lỗi: Chỉ tìm thấy {len(preprocessed_found)} / {len(MIP360_SCENES)} scenes trong Kaggle Input.")
    print("⚠️ Các scene còn thiếu hoặc không chứa thư mục images_8.")
    print("👉 Vui lòng tạo Kaggle Dataset 'MipNerf360-dataset' trước bằng notebook tiền xử lý và add vào notebook này!")
    sys.exit(1)

In [ ]:
# ── Kiểm Tra Tính Đúng Đắn Của Dataset ─────────────────────────────────────────
import os

dataset_root = os.path.join(PROJECT_DIR, "spec-fastgs", "datasets", "mipnerf360")
MIP360_SCENES = ["bicycle", "flowers", "garden", "stump", "treehill", "room", "counter", "kitchen", "bonsai"]
IMAGES = "images_8"

print(f"🔍 Đang kiểm tra thư mục {dataset_root} cho {len(MIP360_SCENES)} scenes x {IMAGES}...")
print("-" * 70)
missing_scenes = []

for scene in MIP360_SCENES:
    scene_dir = os.path.join(dataset_root, scene)
    images_dir = os.path.join(scene_dir, IMAGES)
    
    if not os.path.isdir(scene_dir):
        status = "❌ THIẾU (không tìm thấy thư mục scene)"
        missing_scenes.append(scene)
    elif not os.path.isdir(images_dir):
        status = f"❌ THIẾU (không tìm thấy thư mục {IMAGES}; đang có: {sorted(os.listdir(scene_dir))[:4]})"
        missing_scenes.append(scene)
    else:
        num_images = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        status = f"✅ OK ({num_images} ảnh)"
    print(f"  {scene:<12s} {status}")

print("-" * 70)
if missing_scenes:
    print(f"⚠️ Cảnh báo: Có {len(missing_scenes)} scene thiếu {IMAGES}. Script huấn luyện sẽ tự động bỏ qua chúng.")
else:
    print(f"🎉 Tuyệt vời! Toàn bộ {len(MIP360_SCENES)} scenes đã khớp cấu trúc chuẩn và sẵn sàng!")

## ⚙️ Bước 4: Cài Đặt Môi Trường & Biên Dịch CUDA Submodules

In [ ]:
# ── Cài đặt thư viện Python phụ thuộc ───────────────────────────────────────────
!pip install -q plyfile tqdm websockets openpyxl pandas ninja huggingface_hub

# ── Kiểm tra GPU & Môi trường CUDA ──────────────────────────────────────────────
import torch
import os

gpu_available = torch.cuda.is_available()
print(f"GPU Available in PyTorch: {gpu_available}")
if gpu_available:
    print(f"Active GPU Name         : {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️ CẢNH BÁO: Không tìm thấy GPU hoạt động trong phiên làm việc Kaggle này!")
    print("Huấn luyện 3D Gaussian Splatting cần GPU để chạy và biên dịch submodules.")
    print("Vui lòng bật GPU trong panel bên phải (Settings -> Accelerator -> GPU T4 x2 hoặc GPU P100).\n")

# Tự động phát hiện đường dẫn cài đặt CUDA Toolkit trên Kaggle
cuda_dirs = ["/usr/local/cuda", "/usr/local/cuda-12.2", "/usr/local/cuda-12.1", "/usr/local/cuda-12", "/usr/local/cuda-11.8", "/usr/local/cuda-11", "/opt/conda"]
detected_cuda = None
for d in cuda_dirs:
    if os.path.exists(os.path.join(d, "bin", "nvcc")):
        detected_cuda = d
        break

if detected_cuda:
    print(f"Detected CUDA Toolkit path: {detected_cuda}")
    # Xuất ra file cấu hình môi trường
    with open("/kaggle/working/cuda_env.sh", "w") as f:
        f.write(f"export CUDA_HOME={detected_cuda}\n")
        f.write(f"export PATH={detected_cuda}/bin:$PATH\n")
        f.write(f"export LD_LIBRARY_PATH={detected_cuda}/lib64:$LD_LIBRARY_PATH\n")
else:
    print("❌ Không tìm thấy CUDA Toolkit hoặc NVCC compiler trên hệ thống!")
    print("Hãy chắc chắn bạn đã kích hoạt tùy chọn tăng tốc GPU trong mục Settings của Kaggle.")

In [ ]:
%%bash
# Nạp biến môi trường CUDA tự động phát hiện từ cell trước
if [ -f "/kaggle/working/cuda_env.sh" ]; then
    source /kaggle/working/cuda_env.sh
else
    export CUDA_HOME=/usr/local/cuda
    export PATH=$CUDA_HOME/bin:$PATH
    export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
fi

echo "=== Phiên bản NVCC hiện tại ==="
nvcc --version 2>&1 || echo "nvcc not found (Vui lòng kiểm tra đã kích hoạt GPU trong Kaggle Settings chưa)"
echo "==============================="

PROJECT_DIR="/kaggle/working/thesis-all"
SUBMODULES_DIR="$PROJECT_DIR/FastGS_backup_v2/submodules"

if [ -d "$SUBMODULES_DIR" ]; then
    echo "🔨 Đang biên dịch diff-gaussian-rasterization_fastgs ..."
    cd "$SUBMODULES_DIR/diff-gaussian-rasterization_fastgs"
    rm -rf build dist *.egg-info
    pip install .

    echo "🔨 Đang biên dịch simple-knn ..."
    cd "$SUBMODULES_DIR/simple-knn"
    rm -rf build dist *.egg-info
    pip install .

    echo "🔨 Đang biên dịch fused-ssim ..."
    cd "$SUBMODULES_DIR/fused-ssim"
    rm -rf build dist *.egg-info
    pip install .
else
    echo "❌ Không tìm thấy thư mục submodules tại: $SUBMODULES_DIR"
    echo "Đảm bảo bạn đã chạy đúng Bước 2 để clone repository."
    exit 1
fi

echo "🔬 Đang kiểm tra import submodules ..."
python -c "import diff_gaussian_rasterization_fastgs; import simple_knn; import fused_ssim; print('🎉 Chúc mừng! Đã biên dịch và load CUDA extensions thành công!')"

## 🚀 Bước 5: Chạy Rerun Thực Nghiệm Spec-fastgs Trên Tập images_8

In [ ]:
# ── Cấu Hình Và Chạy Huấn Luyện Song Song Trên 2 GPUs ─────────────────────────────
import os
import subprocess

PROJECT_DIR = "/kaggle/working/thesis-all"
script_path = os.path.join(PROJECT_DIR, "spec-fastgs", "run_parallel_mip360.py")

script_content = """import os
import sys
import subprocess
import time
import queue
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi

PROJECT_DIR = "/kaggle/working/thesis-all"
DATA_ROOT = os.path.join(PROJECT_DIR, "spec-fastgs/datasets/mipnerf360")
IMAGES = "images_8"
OUTPUT_ROOT = os.path.join(PROJECT_DIR, "spec-fastgs/output/mip360_images_8")
HF_TOKEN = os.environ.get('HF_TOKEN', '')
REPO_ID = "DiBiay/spec-fastgs-mipneft360-images8"

SCENES = ["bicycle", "flowers", "garden", "stump", "treehill", "room", "counter", "kitchen", "bonsai"]
GPUS = ["0", "1"]

# Cấu hình biến môi trường CUDA
cuda_dirs = ["/usr/local/cuda", "/usr/local/cuda-12.2", "/usr/local/cuda-12.1", "/usr/local/cuda-12", "/usr/local/cuda-11.8", "/usr/local/cuda-11"]
detected_cuda = None
for d in cuda_dirs:
    if os.path.exists(os.path.join(d, "bin", "nvcc")):
        detected_cuda = d
        break
if detected_cuda:
    os.environ["CUDA_HOME"] = detected_cuda
    os.environ["PATH"] = f"{detected_cuda}/bin:" + os.environ.get("PATH", "")
    os.environ["LD_LIBRARY_PATH"] = f"{detected_cuda}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"CUDA configured using path: {detected_cuda}")

try:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True)
    print(f"✅ HuggingFace Dataset initialized: {REPO_ID}")
except Exception as e:
    print(f"⚠️ Error initializing HuggingFace repository: {e}")

def run_cmd(args, env, scene, log_file):
    cwd = os.path.join(PROJECT_DIR, "spec-fastgs")
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"\\n--- Running command: {' '.join(args)} ---\\n")
        f.flush()
        res = subprocess.run(args, env=env, cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
        if res.returncode != 0:
            raise RuntimeError(f"Command failed with status {res.returncode}: {' '.join(args)}")

def train_scene(scene, gpu_id):
    source_path = os.path.join(DATA_ROOT, scene)
    model_path = os.path.join(OUTPUT_ROOT, scene)
    log_file = os.path.join(OUTPUT_ROOT, f"{scene}_training.log")
    
    os.makedirs(model_path, exist_ok=True)
    print(f"🚀 [GPU {gpu_id}] Bắt đầu huấn luyện scene '{scene}'...")
    
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu_id
    
    if not os.path.isdir(source_path):
        print(f"⚠️ [GPU {gpu_id}] Bỏ qua {scene}: không tìm thấy thư mục {source_path}")
        return False
        
    if os.path.exists(os.path.join(model_path, "results.json")) or os.path.exists(os.path.join(model_path, "results_grouped.json")):
        print(f"⏩ [GPU {gpu_id}] Scene '{scene}' đã hoàn thành trước đó. Tiến hành upload lại...")
        upload_results(scene, model_path)
        return True
        
    start_time = time.time()
    try:
        print(f"🔍 [GPU {gpu_id}] Scene '{scene}': Trích xuất reflection prior...")
        run_cmd([
            sys.executable, "extract_reflection_prior.py",
            "-s", source_path,
            "-i", IMAGES,
            "--ref_prior_method", "tan"
        ], env, scene, log_file)
        
        print(f"🏋️ [GPU {gpu_id}] Scene '{scene}': Huấn luyện Spec-fastgs (30k iterations)...")
        run_cmd([
            sys.executable, "train.py",
            "-s", source_path,
            "-m", model_path,
            "-i", IMAGES,
            "--eval",
            "--iterations", "30000",
            "--densification_interval", "100",
            "--optimizer_type", "default",
            "--asg_degree", "64",
            "--is_real",
            "--is_indoor",
            "--sh_degree", "3",
            "--highfeature_lr", "0.02",
            "--grad_abs_thresh", "0.0004",
            "--specular_start_iter", "3000",
            "--ref_prior_method", "tan",
            "--sh_spec_grad_scale", "0.75",
            "--sh_spec_mask_start", "8000",
            "--sh_spec_mask_threshold", "0.75",
            "--sh_spec_min_metric_count", "2",
            "--use_ref_score",
            "--use_adaptive_prior",
            "--use_sh_spec_mask"
        ], env, scene, log_file)
        
        print(f"🎬 [GPU {gpu_id}] Scene '{scene}': Kết xuất test views...")
        run_cmd([
            sys.executable, "render.py",
            "-m", model_path,
            "--skip_train"
        ], env, scene, log_file)
        
        print(f"📊 [GPU {gpu_id}] Scene '{scene}': Đo đạc chỉ số (metrics)...")
        run_cmd([
            sys.executable, "metrics.py",
            "-m", model_path
        ], env, scene, log_file)
        
        elapsed = time.time() - start_time
        print(f"✅ [GPU {gpu_id}] Hoàn thành scene '{scene}' trong {elapsed/60:.2f} phút!")
        
        upload_results(scene, model_path)
        return True
        
    except Exception as e:
        print(f"❌ [GPU {gpu_id}] Lỗi khi chạy scene '{scene}': {e}")
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"\\n❌ ERROR: {str(e)}\\n")
        return False

def upload_results(scene, model_path):
    print(f"☁️ Đang upload kết quả của scene '{scene}' lên HuggingFace...")
    for attempt in range(3):
        try:
            api.upload_folder(
                folder_path=model_path,
                path_in_repo=f"mip360_images_8/{scene}",
                repo_id=REPO_ID,
                repo_type="dataset"
            )
            print(f"✅ Đã upload thành công scene '{scene}' lên HuggingFace!")
            break
        except Exception as err:
            print(f"⚠️ Thử lại lần {attempt+1}: Lỗi khi upload scene '{scene}': {err}")
            time.sleep(10)

def main():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    gpu_queue = queue.Queue()
    for g in GPUS:
        gpu_queue.put(g)
        
    def worker(scene):
        gpu_id = gpu_queue.get()
        try:
            success = train_scene(scene, gpu_id)
            return scene, success
        finally:
            gpu_queue.put(gpu_id)
            
    print(f"⚙️ Bắt đầu huấn luyện song song {len(SCENES)} scenes trên {len(GPUS)} GPUs...")
    print(f"GPUs khả dụng: {GPUS}")
    
    with ThreadPoolExecutor(max_workers=len(GPUS)) as executor:
        futures = {executor.submit(worker, scene): scene for scene in SCENES}
        for future in as_completed(futures):
            scene = futures[future]
            try:
                scene, success = future.result()
                status_str = "thành công" if success else "thất bại"
                print(f"📢 Trạng thái scene '{scene}': {status_str}")
            except Exception as e:
                print(f"❌ Exception in scene '{scene}': {e}")

if __name__ == '__main__':
    main()"""

os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)
print(f"✅ Đã khởi tạo script chạy song song tại: {script_path}")

# Khởi chạy script chạy song song ở chế độ background
log_file = os.path.join(PROJECT_DIR, "spec-fastgs", "output", "run_parallel_mip360.log")
os.makedirs(os.path.dirname(log_file), exist_ok=True)

print(f"🚀 Đang khởi chạy tiến trình huấn luyện song song...")
print(f"Log tiến trình chính sẽ được ghi vào: {log_file}")

# Chạy ngầm bằng subprocess Popen để không bị lock cell
with open(log_file, "w") as f:
    p = subprocess.Popen(["python", "-u", "run_parallel_mip360.py"], cwd=os.path.join(PROJECT_DIR, "spec-fastgs"), stdout=f, stderr=subprocess.STDOUT)

print(f"✅ Tiến trình đã bắt đầu chạy ngầm với PID: {p.pid}")
print("Bạn hãy chạy cell tiếp theo bên dưới để theo dõi log trực tiếp.")

In [ ]:
# ── Theo Dõi Tiến Trình Train Động ─────────────────────────────────────────────
import time
import os

PROJECT_DIR = "/kaggle/working/thesis-all"
log_path = os.path.join(PROJECT_DIR, "spec-fastgs", "output", "run_parallel_mip360.log")

if os.path.exists(log_path):
    print(f"Đang theo dõi log tiến trình song song: {log_path} (Bấm Stop trong Kaggle để dừng xem log)\n")
    
    # Hiển thị 30 dòng log hiện tại
    with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
        print("".join(f.readlines()[-30:]))
        
    try:
        last_size = os.path.getsize(log_path)
        while True:
            time.sleep(5)
            curr_size = os.path.getsize(log_path)
            if curr_size > last_size:
                with open(log_path, 'r', encoding='utf-8', errors='replace') as f:
                    f.seek(last_size)
                    print(f.read(), end="", flush=True)
                last_size = curr_size
    except KeyboardInterrupt:
        print("\nDừng theo dõi log.")
else:
    print("⚠️ File log chưa được khởi tạo. Đang đợi tiến trình background khởi động...")

## 📊 Bước 6: Tổng Hợp Kết Quả & Xuất File Bảng Biểu Excel (.xlsx)

In [ ]:
# ── Tổng Hợp Kết Quả Thực Nghiệm Vào Bảng Excel (.xlsx) ─────────────────────────
import json
import os
import shutil
import openpyxl

PROJECT_DIR = "/kaggle/working/thesis-all"

def aggregate_to_excel(project_dir):
    template_path = os.path.join(project_dir, "other_job", "mipnerf360_image_2.xlsx")
    output_excel_path = os.path.join(project_dir, "other_job", "mipnerf360_images8.xlsx")
    kaggle_output_path = "/kaggle/working/mipnerf360_images8.xlsx"
    
    # 1. Khởi tạo từ template mẫu (để giữ style, font, border, màu sắc)
    if os.path.exists(template_path):
        print(f"🎯 Tìm thấy file template mẫu tại: {template_path}")
        wb = openpyxl.load_workbook(template_path)
        ws = wb.active
        ws.title = "mipnerf360_images8"
    else:
        print("⚠️ Không tìm thấy file template mẫu. Khởi tạo bảng mới từ đầu...")
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "mipnerf360_images8"
        headers = [
            'Dataset', 'Scene', 'Method', 'PSNR (dB)', 'SSIM', 'LPIPS', 
            'N_init', 'N_final', 'Growth_Ratio', 'Training_Time', 
            'Training_Time_Sec', 'Peak_VRAM_MB', 'FPS'
        ]
        ws.append(headers)
        
    scenes = ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]
    methods = ["3DGS", "FastGS", "Spec-Gaussian", "Spec-fastgs"]
    
    # Thư mục chứa kết quả Spec-fastgs đã train xong cho images_8
    output_root = os.path.join(project_dir, "spec-fastgs", "output", "mip360_images_8")
    
    print("\n--- Bắt đầu quét kết quả Spec-fastgs và cập nhật vào bảng Excel ---")
    
    for i, scene in enumerate(scenes):
        # Dòng bắt đầu của scene hiện tại (mỗi scene chiếm 4 dòng cho 4 methods, bắt đầu từ dòng 2)
        base_row = 2 + 4 * i
        
        # Nếu khởi tạo mới, điền Dataset, Scene và Method
        if not os.path.exists(template_path):
            ws.cell(row=base_row, column=1, value="mipnerf360")
            ws.cell(row=base_row, column=2, value=scene)
            for m_idx, method in enumerate(methods):
                ws.cell(row=base_row + m_idx, column=3, value=method)
                
        # Dòng cho Spec-fastgs nằm ở dòng thứ 4 của block scene hiện tại (tương ứng offset +3)
        spec_fastgs_row = base_row + 3
        scene_output_dir = os.path.join(output_root, scene)
        results_json_path = os.path.join(scene_output_dir, "results.json")
        train_info_path = os.path.join(scene_output_dir, "train_info.json")
        
        # 2. Xóa trắng các chỉ số của 3DGS, FastGS, Spec-Gaussian (offset +0, +1, +2)
        # vì chúng ta chỉ đang thực nghiệm và đo đạc lại Spec-fastgs trên images_8
        for offset in [0, 1, 2]:
            for col in range(4, 14):
                ws.cell(row=base_row + offset, column=col, value=None)
                
        # 3. Đọc dữ liệu Spec-fastgs và cập nhật
        if os.path.exists(results_json_path) and os.path.exists(train_info_path):
            try:
                with open(results_json_path, 'r', encoding='utf-8') as f:
                    results = json.load(f)
                with open(train_info_path, 'r', encoding='utf-8') as f:
                    train_info = json.load(f)
                
                # Tìm key phương pháp động (thường là 'ours_30000')
                method_key = list(results.keys())[0] if results else None
                if method_key:
                    res_metrics = results[method_key]
                    
                    psnr = res_metrics.get("PSNR")
                    ssim = res_metrics.get("SSIM")
                    lpips = res_metrics.get("LPIPS")
                    fps = res_metrics.get("FPS")
                    
                    n_init = train_info.get("initial_gaussians")
                    n_final = train_info.get("final_gaussians")
                    growth_ratio = round(n_final / n_init, 2) if (n_final and n_init) else None
                    train_time_formatted = train_info.get("training_time_formatted")
                    train_time_sec = train_info.get("training_time_seconds")
                    peak_vram = train_info.get("peak_vram_mib")
                    
                    # Điền các giá trị vào dòng Spec-fastgs
                    ws.cell(row=spec_fastgs_row, column=4, value=psnr)
                    ws.cell(row=spec_fastgs_row, column=5, value=ssim)
                    ws.cell(row=spec_fastgs_row, column=6, value=lpips)
                    ws.cell(row=spec_fastgs_row, column=7, value=n_init)
                    ws.cell(row=spec_fastgs_row, column=8, value=n_final)
                    ws.cell(row=spec_fastgs_row, column=9, value=growth_ratio)
                    ws.cell(row=spec_fastgs_row, column=10, value=train_time_formatted)
                    ws.cell(row=spec_fastgs_row, column=11, value=train_time_sec)
                    ws.cell(row=spec_fastgs_row, column=12, value=peak_vram)
                    ws.cell(row=spec_fastgs_row, column=13, value=fps)
                    
                    print(f"  ✅ Scene {scene:<12s}: PSNR={psnr:.4f} | SSIM={ssim:.4f} | LPIPS={lpips:.4f} | Final_Gauss={n_final}")
            except Exception as e:
                print(f"  ❌ Lỗi khi đọc kết quả scene {scene}: {e}")
        else:
            print(f"  ⚠️ Cảnh báo: Chưa tìm thấy kết quả của scene '{scene}' tại {scene_output_dir}")
            # Xóa các cell kết quả nếu không có kết quả
            for col in range(4, 14):
                ws.cell(row=spec_fastgs_row, column=col, value=None)
                
    # Kích hoạt hiển thị gridlines trong Excel
    try:
        ws.views.sheetView[0].showGridLines = True
    except Exception:
        pass
        
    wb.save(output_excel_path)
    print("----------------------------------------------------------------")
    print(f"🎉 Hoàn tất! Bảng Excel đã được cập nhật và lưu tại:\n  {output_excel_path}")
    
    # 4. Sao chép bảng kết quả ra thư mục output gốc của Kaggle để tải xuống từ giao diện UI
    try:
        shutil.copy(output_excel_path, kaggle_output_path)
        print(f"📌 Đã đồng bộ file Excel ra thư mục Output Kaggle: {kaggle_output_path}")
        print("   -> Bạn có thể tải trực tiếp file này về máy tính ở mục 'Data -> Output' bên cạnh.")
    except Exception as copy_err:
        print(f"   ❌ Lỗi khi sao chép file kết quả ra Output Kaggle: {copy_err}")

# Chạy tổng hợp kết quả
aggregate_to_excel(PROJECT_DIR)